In [1]:
import numpy as np

### Get Sentences using Ingestion tool

In [2]:
from components import *

sentences = ingest_using_url("https://jamesclear.com/saying-no")
print(sentences)

combined = get_scores(sentences)
print(combined)
print(f'Number of scores: {len(combined['combined'])}')

features = combine_features(
    sentences=sentences,
    textrank_scores_arr=combined['textrank'],
    centroid_scores_arr=combined['centroid'],
    domain='general'
)

scores = features['combined']

THRESHOLD   = 0.60
deduped_sents, deduped_scores, kept_idx = deduplicate(
    sentences, scores, threshold=THRESHOLD, hnsw_min_sentences=20
)

c:\Users\Justin\Documents\2026 projects\python\textSummariserExample\summ\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8565.91it/s]


['The Ultimate Productivity Hack is Saying No - James Clear The Ultimate Productivity Hack is Saying No written by James Clear Decision Making Focus Life Lessons The ultimate productivity hack is saying no.', 'Not doing something will always be faster than doing it.', 'This statement reminds me of the old computer programming saying, “Remember that there is no code faster than no code.” 1 The same philosophy applies in other areas of life.', 'For example, there is no meeting that goes faster than not having a meeting at all.', "This is not to say you should never attend another meeting, but the truth is that we say yes to many things we don't actually want to do.", "There are many meetings held that don't need to be held.", 'There is a lot of code written that could be deleted.', "How often do people ask you to do something and you just reply, “Sure thing.” Three days later, you're overwhelmed by how much is on your to-do list.", 'We become frustrated by our obligations even though we 

### Why do we need a length budget?
#### Without a budget, a summariser might include too many sentences
#### (barely shorter than the original) or too few (loses key information).
####
#### The budget answers: "How many sentences should the summary contain?"

### We balance three things:

####   1. COMPRESSION RATIO
####      The summary should be a fraction of the original.
####     e.g. ratio=0.3 means: if the doc has 20 sentences, keep ~6.
####      Good for maintaining proportional coverage.

####   2. WORD COUNT CEILING
####      Hard cap on total words — useful when the summary has a
####      specific display constraint (e.g. 150-word abstract, tweet, etc.)
####     We simulate adding sentences by score until we hit the cap.

####   3. SCORE THRESHOLD
####      Only include sentences above a minimum quality score.
####      Prevents low-quality sentences sneaking in just to fill the budget.
####      e.g. threshold=0.3 means: sentences scoring below 0.3 are excluded
####      even if the ratio budget would allow more.

#### Final budget = most restrictive of all three constraints.

### 1. Compression ratio

In [3]:
#5a COMPRESSION RATIO BUDGET

def ratio_budget(n_sentences: int,
                 compression_ratio: float = 0.3,
                 min_sentences: int = 1,
                 max_sentences: int = 20) -> int:
    """
    Compute how many sentences to keep based on a compression ratio.
 
    Parameters
    ----------
    n_sentences      : total number of sentences after deduplication
    compression_ratio: fraction of sentences to keep (0.0 to 1.0)
                         0.1 = very aggressive (keep 10%)
                         0.3 = moderate (keep 30%) — good default
                         0.5 = light (keep 50%)
    min_sentences    : always keep at least this many (floor)
    max_sentences    : never keep more than this many (ceiling)
 
    Returns
    -------
    budget : int
 
    Examples
    --------
    20 sentences, ratio=0.30 → keep 6
    50 sentences, ratio=0.20 → keep 10
     5 sentences, ratio=0.30 → keep 2 (rounds up from 1.5, then hits min=1)
    """
    raw    = n_sentences * compression_ratio
    budget = max(min_sentences, round(raw))
    budget = min(budget, max_sentences)
    budget = min(budget, n_sentences)   # can't keep more than we have
    return budget

In [6]:
print("\n── 5a. Compression Ratio Budget ──")
for ratio in [0.20, 0.30, 0.40]:
    b = ratio_budget(len(sentences), ratio)
    print(f"  ratio={ratio:.0%}  →  budget={b} sentences  "
            f"({ratio:.0%} × {len(sentences)} = {len(sentences)*ratio:.1f}, rounded)")


── 5a. Compression Ratio Budget ──
  ratio=20%  →  budget=19 sentences  (20% × 94 = 18.8, rounded)
  ratio=30%  →  budget=20 sentences  (30% × 94 = 28.2, rounded)
  ratio=40%  →  budget=20 sentences  (40% × 94 = 37.6, rounded)


### 2. Word count

In [7]:
# 5b. WORD COUNT BUDGET

def word_count_budget(sentences: list[str],
                      scores: np.ndarray,
                      max_words: int = 150) -> int:
    """
    Compute how many sentences fit within a word count ceiling.
 
    Strategy: greedily add sentences in descending score order,
    counting words until the next sentence would exceed max_words.
 
    Parameters
    ----------
    sentences : deduplicated sentences from Step 4
    scores    : their combined scores from Step 3
    max_words : hard word count limit for the whole summary
 
    Returns
    -------
    budget : int — number of sentences that fit within max_words
 
    Examples
    --------
    max_words=150, sentences averaging 20 words → budget ≈ 7
    max_words=50,  sentences averaging 20 words → budget ≈ 2
    """
    # Sort by score descending — best sentences get priority
    sorted_idx = np.argsort(scores)[::-1]
 
    total_words = 0
    budget      = 0
 
    for idx in sorted_idx:
        wc = len(sentences[idx].split())
 
        # Would adding this sentence exceed the word limit?
        if total_words + wc > max_words:
            break                  # stop — next sentence doesn't fit
 
        total_words += wc
        budget      += 1
 
    return max(1, budget)          # always allow at least 1 sentence

In [8]:
print("\n── 5b. Word Count Budget ──")
print("  Adds sentences in score order until word limit is hit:\n")
sorted_idx  = np.argsort(scores)[::-1]
running_wc  = 0
print(f"  {'Rank':<5} {'Sent':>5} {'Score':>6} {'Words':>6} "
      f"{'Running Total':>14} {'Status'}")
print("  " + "-" * 58)
for rank, idx in enumerate(sorted_idx, 1):
    wc     = len(sentences[idx].split())
    fits   = (running_wc + wc) <= 150
    status = "✓ included" if fits else "✗ exceeds limit"
    if fits:
        running_wc += wc
    print(f"  {rank:<5} S{idx+1:<4} {scores[idx]:>6.2f} {wc:>6} "
          f"{running_wc:>14}  {status}")
wc_b = word_count_budget(sentences, scores, max_words=150)
print(f"\n  → Word count budget (max_words=150): {wc_b} sentences")


── 5b. Word Count Budget ──
  Adds sentences in score order until word limit is hit:

  Rank   Sent  Score  Words  Running Total Status
  ----------------------------------------------------------
  1     S28     1.00     51             51  ✓ included
  2     S30     0.91      8             59  ✓ included
  3     S5      0.89     28             87  ✓ included
  4     S12     0.87     18            105  ✓ included
  5     S27     0.85     12            117  ✓ included
  6     S29     0.84     11            128  ✓ included
  7     S39     0.77     25            128  ✗ exceeds limit
  8     S26     0.77     12            140  ✓ included
  9     S60     0.77     51            140  ✗ exceeds limit
  10    S15     0.76     19            140  ✗ exceeds limit
  11    S55     0.71     36            140  ✗ exceeds limit
  12    S1      0.70     33            140  ✗ exceeds limit
  13    S40     0.69     33            140  ✗ exceeds limit
  14    S13     0.68     29            140  ✗ exceeds lim

### 3. Score threshold

In [9]:
# 5c. SCORE THRESHOLD BUDGET

def score_threshold_budget(scores: np.ndarray,
                            threshold: float = 0.25) -> int:
    """
    Count how many sentences score above the minimum quality threshold.
 
    Any sentence below the threshold is considered too low-quality to
    include, regardless of what the ratio or word count budget allows.
 
    Parameters
    ----------
    scores    : combined scores from Step 3, shape (n,)
    threshold : minimum score to qualify (0.0 to 1.0)
                  0.10 = very permissive (almost everything qualifies)
                  0.25 = moderate — good default
                  0.50 = strict (only clearly important sentences)
 
    Returns
    -------
    budget : int — number of sentences above the threshold
 
    Examples
    --------
    scores=[0.9, 0.7, 0.5, 0.3, 0.1], threshold=0.25
        → 4 sentences qualify (0.9, 0.7, 0.5, 0.3 are all >= 0.25)
    """
    qualifying = int(np.sum(scores >= threshold))
    return max(1, qualifying)      # always allow at least 1

In [10]:
print("\n── 5c. Score Threshold Budget ──")
THRESHOLD = 0.25
print(f"  threshold = {THRESHOLD}  (exclude sentences scoring below this)\n")
print(f"  {'Sent':>5} {'Score':>7}  {'Status'}")
print("  " + "-" * 30)
for i, (sent, score) in enumerate(zip(sentences, scores)):
    status = f"✓ qualifies" if score >= THRESHOLD else "✗ below threshold"
    print(f"  S{i+1:<4} {score:>7.2f}  {status}")
th_b = score_threshold_budget(scores, threshold=THRESHOLD)
print(f"\n  → Score threshold budget: {th_b} sentences qualify")


── 5c. Score Threshold Budget ──
  threshold = 0.25  (exclude sentences scoring below this)

   Sent   Score  Status
  ------------------------------
  S1       0.70  ✓ qualifies
  S2       0.59  ✓ qualifies
  S3       0.66  ✓ qualifies
  S4       0.52  ✓ qualifies
  S5       0.89  ✓ qualifies
  S6       0.55  ✓ qualifies
  S7       0.29  ✓ qualifies
  S8       0.64  ✓ qualifies
  S9       0.51  ✓ qualifies
  S10      0.51  ✓ qualifies
  S11      0.49  ✓ qualifies
  S12      0.87  ✓ qualifies
  S13      0.68  ✓ qualifies
  S14      0.59  ✓ qualifies
  S15      0.76  ✓ qualifies
  S16      0.46  ✓ qualifies
  S17      0.51  ✓ qualifies
  S18      0.51  ✓ qualifies
  S19      0.36  ✓ qualifies
  S20      0.44  ✓ qualifies
  S21      0.56  ✓ qualifies
  S22      0.53  ✓ qualifies
  S23      0.54  ✓ qualifies
  S24      0.59  ✓ qualifies
  S25      0.35  ✓ qualifies
  S26      0.77  ✓ qualifies
  S27      0.85  ✓ qualifies
  S28      1.00  ✓ qualifies
  S29      0.84  ✓ qualifies
  S30   

### 4. Combine contraints

In [11]:
# 5d. COMBINE ALL CONSTRAINTS

def compute_budget(sentences: list[str],
                   scores: np.ndarray,
                   compression_ratio: float = 0.3,
                   max_words: int = 150,
                   score_threshold: float = 0.25,
                   min_sentences: int = 1,
                   max_sentences: int = 20,
                   verbose: bool = True) -> dict:
    """
    Run all three constraints and return the most restrictive budget.
 
    Parameters
    ----------
    sentences         : deduplicated sentences from Step 4
    scores            : their combined scores from Step 3
    compression_ratio : fraction of sentences to keep
    max_words         : hard word count ceiling for the summary
    score_threshold   : minimum score to qualify for inclusion
    min_sentences     : absolute floor on budget
    max_sentences     : absolute ceiling on budget
    verbose           : print breakdown of each constraint
 
    Returns
    -------
    dict with:
        budget          : int  — final number of sentences to select  ← use this
        ratio_b         : int  — ratio-based budget
        word_count_b    : int  — word-count-based budget
        threshold_b     : int  — score-threshold-based budget
        binding         : str  — which constraint was most restrictive
        qualifying_scores: np.ndarray — mask of sentences above threshold
    """
    n = len(sentences)
 
    # Run each constraint independently
    ratio_b      = ratio_budget(n, compression_ratio, min_sentences, max_sentences)
    word_count_b = word_count_budget(sentences, scores, max_words)
    threshold_b  = score_threshold_budget(scores, score_threshold)
 
    # Most restrictive wins
    final = min(ratio_b, word_count_b, threshold_b)
    final = max(final, min_sentences)   # never go below the floor
 
    # Identify which constraint is binding (most restrictive)
    budgets = {
        "compression_ratio": ratio_b,
        "word_count":        word_count_b,
        "score_threshold":   threshold_b,
    }
    binding = min(budgets, key=budgets.get)
 
    # Which sentences individually qualify by score
    qualifying_mask = scores >= score_threshold
 
    if verbose:
        total_words = sum(len(s.split()) for s in sentences)
        print(f"  Input          : {n} sentences, {total_words} total words")
        print(f"  Ratio budget   : {ratio_b}  "
              f"({compression_ratio:.0%} of {n} sentences)")
        print(f"  Word-count bgt : {word_count_b}  "
              f"(best sentences within {max_words} words)")
        print(f"  Threshold bgt  : {threshold_b}  "
              f"({int(qualifying_mask.sum())} sentences ≥ {score_threshold} score)")
        print(f"  ─────────────────────────────")
        print(f"  Final budget   : {final}  ← binding constraint: {binding}")
 
    return {
        "budget":            final,
        "ratio_b":           ratio_b,
        "word_count_b":      word_count_b,
        "threshold_b":       threshold_b,
        "binding":           binding,
        "qualifying_mask":   qualifying_mask,
    }

In [12]:
print("\n── 5d. Final Budget (most restrictive constraint wins) ──\n")
result = compute_budget(
    sentences, scores,
    compression_ratio=0.30,
    max_words=150,
    score_threshold=0.25,
)
print(f"\n  The {result['binding']} constraint is binding.")
print(f"  Pass budget={result['budget']} to Step 6 (MMR selection).")


── 5d. Final Budget (most restrictive constraint wins) ──

  Input          : 94 sentences, 1861 total words
  Ratio budget   : 20  (30% of 94 sentences)
  Word-count bgt : 6  (best sentences within 150 words)
  Threshold bgt  : 86  (86 sentences ≥ 0.25 score)
  ─────────────────────────────
  Final budget   : 6  ← binding constraint: word_count

  The word_count constraint is binding.
  Pass budget=6 to Step 6 (MMR selection).


### Helper function

In [13]:
# HELPER: show how budget changes across settings

def budget_sensitivity(sentences: list[str],
                        scores: np.ndarray) -> None:
    """
    Print a table showing how each parameter affects the final budget.
    Useful for tuning before you run the full pipeline.
    """
    print("\n── Sensitivity: Compression Ratio ──")
    print(f"  {'Ratio':>8}  {'Budget':>8}  {'Est. words':>12}")
    print("  " + "-" * 32)
    for ratio in [0.10, 0.20, 0.30, 0.40, 0.50]:
        b = ratio_budget(len(sentences), ratio)
        # Estimate words: take top-b sentences by score, sum their words
        top_idx   = np.argsort(scores)[::-1][:b]
        est_words = sum(len(sentences[i].split()) for i in top_idx)
        print(f"  {ratio:>8.0%}  {b:>8}  {est_words:>12}")
 
    print("\n── Sensitivity: Max Words ──")
    print(f"  {'Max words':>10}  {'Budget':>8}")
    print("  " + "-" * 22)
    for mw in [50, 75, 100, 150, 200, 300]:
        b = word_count_budget(sentences, scores, max_words=mw)
        print(f"  {mw:>10}  {b:>8}")
 
    print("\n── Sensitivity: Score Threshold ──")
    print(f"  {'Threshold':>10}  {'Qualifying':>12}  {'Sentences above'}") 
    print("  " + "-" * 55)
    for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]:
        b    = score_threshold_budget(scores, threshold=t)
        qual = [f"S{i+1}({scores[i]:.2f})" for i in range(len(scores))
                if scores[i] >= t]
        print(f"  {t:>10.2f}  {b:>12}  {', '.join(qual)}")

In [14]:
print("\n── 5e. Sensitivity Analysis ──")
budget_sensitivity(sentences, scores)


── 5e. Sensitivity Analysis ──

── Sensitivity: Compression Ratio ──
     Ratio    Budget    Est. words
  --------------------------------
       10%         9           216
       20%        19           571
       30%        20           599
       40%        20           599
       50%        20           599

── Sensitivity: Max Words ──
   Max words    Budget
  ----------------------
          50         1
          75         2
         100         3
         150         6
         200         8
         300        11

── Sensitivity: Score Threshold ──
   Threshold    Qualifying  Sentences above
  -------------------------------------------------------
        0.10            90  S1(0.70), S2(0.59), S3(0.66), S4(0.52), S5(0.89), S6(0.55), S7(0.29), S8(0.64), S9(0.51), S10(0.51), S11(0.49), S12(0.87), S13(0.68), S14(0.59), S15(0.76), S16(0.46), S17(0.51), S18(0.51), S19(0.36), S20(0.44), S21(0.56), S22(0.53), S23(0.54), S24(0.59), S25(0.35), S26(0.77), S27(0.85), S28(1.00), S29(

In [15]:
print("\n── Use Case Presets ──")
presets = {
    "Tweet / social post":    dict(compression_ratio=0.10, max_words=30,  score_threshold=0.50),
    "Email preview (3 lines)":dict(compression_ratio=0.20, max_words=60,  score_threshold=0.40),
    "Standard summary":       dict(compression_ratio=0.30, max_words=150, score_threshold=0.25),
    "Detailed summary":       dict(compression_ratio=0.40, max_words=250, score_threshold=0.20),
    "Long-form digest":       dict(compression_ratio=0.50, max_words=400, score_threshold=0.15),
}
print(f"\n  {'Use case':<28} {'Ratio':>6} {'MaxW':>5} {'Thresh':>7} {'Budget':>7}")
print("  " + "-" * 60)
for name, params in presets.items():
    r = compute_budget(sentences, scores, verbose=False, **params)
    print(f"  {name:<28} "
          f"{params['compression_ratio']:>6.0%} "
          f"{params['max_words']:>5} "
          f"{params['score_threshold']:>7.2f} "
          f"{r['budget']:>7}")


── Use Case Presets ──

  Use case                      Ratio  MaxW  Thresh  Budget
  ------------------------------------------------------------
  Tweet / social post             10%    30    0.50       1
  Email preview (3 lines)         20%    60    0.40       2
  Standard summary                30%   150    0.25       6
  Detailed summary                40%   250    0.20      10
  Long-form digest                50%   400    0.15      15
